# Pipeline Final Test



In [6]:
# ============================================================
# AlgaeScan Python pipeline
# Machine learning-driven identification of microalgae
# Full-spectrum flow cytometry
# ============================================================

from __future__ import annotations

# ============================================================
# Standard library
# ============================================================

import itertools
import random
import warnings
from pathlib import Path
import json


# ============================================================
# Data handling and model persistence
# ============================================================

import joblib
import numpy as np
import pandas as pd


# ============================================================
# Machine learning models
# ============================================================

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import OneClassSVM


# ============================================================
# Preprocessing
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# Model selection and cross-validation
# ============================================================

from sklearn.model_selection import GridSearchCV, StratifiedKFold


# ============================================================
# Model evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

In [19]:
# ============================================================
# Pipeline configuration
# ============================================================

# Reproducibility
RANDOM_STATE = 123

# Reduce non-critical warnings
warnings.filterwarnings("ignore")


# ============================================================
# Input and output paths
# ============================================================

# Change the path based on your dataset to analyze

# SOURCE_DATASET_PATH controls the path to a dataset that needs to be downsampled
SOURCE_DATASET_PATH = Path(
    "../Pipeline/A02_002_dataset_02_2024/A02_002_m1m2m3noise_bal10000.csv"
)

# SMALL_DATASET_PATH controls the path to a dataset that is already downsampled
SMALL_DATASET_PATH = Path(
    "../Pipeline/small_dataset.csv"
)

OUTPUT_DIR = Path("../../results_python_pipeline/outputs")

# ============================================================
# Dataset configuration
# ============================================================

# Set to False if you plan to use the dataset indicated in SMALL_DATASET_PATH, otherwise keep it TRUE
# It means performing downsampling (TRUE) or no (FAlSE)
CREATE_SMALL_DATASET = True 
N_SAMPLES = 1000
SAMPLING_STRATEGY = "stratified"

LABEL_COLUMN = "type"
SPECIES_COLUMN = "Species"
NON_ALGAE_LABEL = "non-algae"


# ============================================================
# Model configuration
# ============================================================

CV_FOLDS = 5

RUN_MODELS = [
    "A_rf",
    "A_if",
    "A_ocsvm",
    "B_rf",
    "B_mlr",
]


# ============================================================
# Columns excluded from model training
# ============================================================

MODEL_A_DROP_COLUMNS = ["Species", "Class", "Exp"]
MODEL_B_DROP_COLUMNS = ["Class", "Exp", "type"]

In [20]:
print("Source dataset:")
print(SOURCE_DATASET_PATH.resolve())
print("Exists:", SOURCE_DATASET_PATH.exists())

print("\nSmall dataset:")
print(SMALL_DATASET_PATH.resolve())

print("\nOutput directory:")
print(OUTPUT_DIR.resolve())

Source dataset:
/workspace/Aggregated data for paper/Pipeline/A02_002_dataset_02_2024/A02_002_m1m2m3noise_bal10000.csv
Exists: True

Small dataset:
/workspace/Aggregated data for paper/Pipeline/small_dataset.csv

Output directory:
/workspace/results_python_pipeline/outputs


In [21]:
# ============================================================
# Load AlgaeScan pipeline functions
# ============================================================

%run -i pipeline_final.py


# ============================================================
# Check imported pipeline functions
# ============================================================

functions_to_check = [
    create_small_dataset,
    load_dataset,
    make_binary_label,
    filter_algae_data,
    Train_model_A_rf,
    Predict_model_A_rf,
    Train_model_A_if,
    Predict_model_A_if,
    Train_model_A_ocsvm,
    Predict_model_A_ocsvm,
    train_model_B_rf,
    Predict_model_B_rf,
    train_model_B_mlr,
    Predict_model_B_mlr,
    evaluate_predictions,
    save_model,
    save_prediction_results,
    save_metrics,
    add_metric_row,
    print_metric_row,
]

for func in functions_to_check:
    print(f"{func.__name__}: imported correctly")

create_small_dataset: imported correctly
load_dataset: imported correctly
make_binary_label: imported correctly
filter_algae_data: imported correctly
Train_model_A_rf: imported correctly
Predict_model_A_rf: imported correctly
Train_model_A_if: imported correctly
Predict_model_A_if: imported correctly
Train_model_A_ocsvm: imported correctly
Predict_model_A_ocsvm: imported correctly
train_model_B_rf: imported correctly
Predict_model_B_rf: imported correctly
train_model_B_mlr: imported correctly
Predict_model_B_mlr: imported correctly
evaluate_predictions: imported correctly
save_model: imported correctly
save_prediction_results: imported correctly
save_metrics: imported correctly
add_metric_row: imported correctly
print_metric_row: imported correctly


In [22]:
# ============================================================
# Initialize pipeline
# ============================================================

# Set random seed for reproducibility
set_seed(RANDOM_STATE)

# Define output subdirectories
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "predictions"

# Initialize container for model evaluation metrics
metrics_rows = []

In [23]:
# ============================================================
# Create or load test dataset
# ============================================================

if CREATE_SMALL_DATASET:
    print("Creating downsampled and formatted dataset...")

    df = create_small_dataset(
        input_path=SOURCE_DATASET_PATH,
        output_path=SMALL_DATASET_PATH,
        n_samples=N_SAMPLES,
        label_column=LABEL_COLUMN,
        sampling_strategy=SAMPLING_STRATEGY,
        random_state=RANDOM_STATE,
    )

else:
    print("Loading existing formatted dataset...")

    df = load_dataset(SMALL_DATASET_PATH)


# ============================================================
# Dataset summary
# ============================================================

print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print("Models selected:", ", ".join(RUN_MODELS))

Creating small dataset...
Rows: 1000
Columns: 77
Models selected: A_rf, A_if, A_ocsvm, B_rf, B_mlr


In [24]:
# ============================================================
# Prepare Model A ground truth
# ============================================================

y_A = make_binary_label(
    df[LABEL_COLUMN],
    non_algae_label=NON_ALGAE_LABEL,
)

print("Model A labels:")
print(y_A.value_counts())

Model A labels:
type
algae        778
non-algae    222
Name: count, dtype: int64


In [25]:
# ============================================================
# Model A - Random Forest
# ============================================================

if "A_rf" in RUN_MODELS:
    print("Training Model A - Random Forest...")

    # Train model
    model_A_rf = Train_model_A_rf(
        train_data=df,
        var_downs=LABEL_COLUMN,
        drop_columns=MODEL_A_DROP_COLUMNS,
        cv=CV_FOLDS,
    )

    # Generate predictions
    preds_A_rf = Predict_model_A_rf(
        df,
        model_A_rf,
    )

    # Evaluate predictions
    metrics_A_rf = evaluate_predictions(
        y_A,
        preds_A_rf,
    )

    # Store compact metrics
    row_A_rf = add_metric_row(
        "A_rf",
        metrics_A_rf,
        preds_A_rf,
    )

    metrics_rows.append(row_A_rf)

    # Display results
    print_metric_row(row_A_rf)

    # Save model and predictions
    save_model(
        model_A_rf,
        MODEL_DIR / "model_A_rf.pkl",
    )

    save_prediction_results(
        preds_A_rf,
        y_A,
        "A_rf",
        PREDICTION_DIR,
    )

Training Model A - Random Forest...
A_rf: accuracy=1.0000, balanced_accuracy=1.0000, macro_f1=1.0000, predictions=1000


In [26]:
# ============================================================
# Model A - Isolation Forest
# ============================================================

if "A_if" in RUN_MODELS:
    print("Training Model A - Isolation Forest...")

    model_A_if = Train_model_A_if(
        train_data=df,
        var_downs=LABEL_COLUMN,
        drop_columns=MODEL_A_DROP_COLUMNS,
        cv=CV_FOLDS,
    )

    preds_A_if = Predict_model_A_if(df, model_A_if)

    metrics_A_if = evaluate_predictions(
        y_A,
        preds_A_if,
    )

    row_A_if = add_metric_row(
        "A_if",
        metrics_A_if,
        preds_A_if,
    )

    metrics_rows.append(row_A_if)

    print_metric_row(row_A_if)

    print(
        f"A_if algae training rows: "
        f"{model_A_if.training_rows_}"
    )

    save_model(
        model_A_if,
        MODEL_DIR / "model_A_if.pkl",
    )

    save_prediction_results(
        preds_A_if,
        y_A,
        "A_if",
        PREDICTION_DIR,
    )

Training Model A - Isolation Forest...
A_if: accuracy=0.9060, balanced_accuracy=0.9138, macro_f1=0.8757, predictions=1000
A_if algae training rows: 778


In [27]:
# ============================================================
# Model A - One-Class SVM
# ============================================================

if "A_ocsvm" in RUN_MODELS:
    print("Training Model A - One-Class SVM...")

    # Train model
    model_A_ocsvm = Train_model_A_ocsvm(
        train_data=df,
        var_downs=LABEL_COLUMN,
        drop_columns=MODEL_A_DROP_COLUMNS,
        cv=CV_FOLDS,
    )

    # Generate predictions
    preds_A_ocsvm = Predict_model_A_ocsvm(
        df,
        model_A_ocsvm,
    )

    # Evaluate predictions
    metrics_A_ocsvm = evaluate_predictions(
        y_A,
        preds_A_ocsvm,
    )

    # Store compact metrics
    row_A_ocsvm = add_metric_row(
        "A_ocsvm",
        metrics_A_ocsvm,
        preds_A_ocsvm,
    )

    metrics_rows.append(row_A_ocsvm)

    # Display results
    print_metric_row(row_A_ocsvm)

    print(
        f"A_ocsvm algae training rows: "
        f"{model_A_ocsvm.training_rows_}"
    )

    # Save model and predictions
    save_model(
        model_A_ocsvm,
        MODEL_DIR / "model_A_ocsvm.pkl",
    )

    save_prediction_results(
        preds_A_ocsvm,
        y_A,
        "A_ocsvm",
        PREDICTION_DIR,
    )

Training Model A - One-Class SVM...
A_ocsvm: accuracy=0.9110, balanced_accuracy=0.9251, macro_f1=0.8830, predictions=1000
A_ocsvm algae training rows: 778


In [28]:
# ============================================================
# Prepare Model B dataset
# ============================================================

# Model B uses algae events only
algae_df = filter_algae_data(
    df,
    species_col=SPECIES_COLUMN,
    non_algae_label=NON_ALGAE_LABEL,
)

# Ground-truth species labels
y_B = algae_df[SPECIES_COLUMN]

print(f"Model B rows: {len(algae_df)}")
print(f"Number of species: {y_B.nunique()}")

print("\nSpecies distribution:")
print(y_B.value_counts())

Model B rows: 778
Number of species: 18

Species distribution:
Species
RCC6336     62
RCC950      55
RCC10364    50
RCC6537     50
RCC88       49
RCC539      48
RCC1435     46
RCC128      46
RCC1507     44
RCC10619    43
RCC827      43
RCC6737     42
RCC2555     39
RCC407      37
RCC3042     37
RCC8        36
RCC69       26
RCC1511     25
Name: count, dtype: int64


In [29]:
# ============================================================
# Model B - Random Forest
# ============================================================

if "B_rf" in RUN_MODELS:
    print("Training Model B - Random Forest...")

    # Train model
    model_B_rf = train_model_B_rf(
        train_data=df,
        var_downs=SPECIES_COLUMN,
        drop_columns=MODEL_B_DROP_COLUMNS,
        non_algae_label=NON_ALGAE_LABEL,
        cv=CV_FOLDS,
    )

    # Generate predictions on algae-only data
    preds_B_rf = Predict_model_B_rf(
        algae_df,
        model_B_rf,
    )

    # Evaluate predictions
    metrics_B_rf = evaluate_predictions(
        y_B,
        preds_B_rf,
    )

    # Store compact metrics
    row_B_rf = add_metric_row(
        "B_rf",
        metrics_B_rf,
        preds_B_rf,
    )

    metrics_rows.append(row_B_rf)

    # Display results
    print_metric_row(row_B_rf)

    print(
        f"B_rf algae training rows: "
        f"{model_B_rf.training_rows_}"
    )

    # Save model and predictions
    save_model(
        model_B_rf,
        MODEL_DIR / "model_B_rf.pkl",
    )

    save_prediction_results(
        preds_B_rf,
        y_B,
        "B_rf",
        PREDICTION_DIR,
    )

Training Model B - Random Forest...
B_rf: accuracy=0.9974, balanced_accuracy=0.9977, macro_f1=0.9975, predictions=778
B_rf algae training rows: 778


In [30]:
# ============================================================
# Model B - Multinomial Logistic Regression
# ============================================================

if "B_mlr" in RUN_MODELS:
    print("Training Model B - Multinomial Logistic Regression...")

    # Train model
    model_B_mlr = train_model_B_mlr(
        train_data=df,
        var_downs=SPECIES_COLUMN,
        drop_columns=MODEL_B_DROP_COLUMNS,
        non_algae_label=NON_ALGAE_LABEL,
        decay=0.1,
        cv=CV_FOLDS,
    )

    # Generate predictions on algae-only data
    preds_B_mlr = Predict_model_B_mlr(
        algae_df,
        model_B_mlr,
    )

    # Evaluate predictions
    metrics_B_mlr = evaluate_predictions(
        y_B,
        preds_B_mlr,
    )

    # Store compact metrics
    row_B_mlr = add_metric_row(
        "B_mlr",
        metrics_B_mlr,
        preds_B_mlr,
    )

    metrics_rows.append(row_B_mlr)

    # Display results
    print_metric_row(row_B_mlr)

    print(
        f"B_mlr algae training rows: "
        f"{model_B_mlr.training_rows_}"
    )

    # Save model and predictions
    save_model(
        model_B_mlr,
        MODEL_DIR / "model_B_mlr.pkl",
    )

    save_prediction_results(
        preds_B_mlr,
        y_B,
        "B_mlr",
        PREDICTION_DIR,
    )

Training Model B - Multinomial Logistic Regression...
B_mlr: accuracy=1.0000, balanced_accuracy=1.0000, macro_f1=1.0000, predictions=778
B_mlr algae training rows: 778


In [31]:
# ============================================================
# Save model evaluation summary
# ============================================================

metric_files = save_metrics(
    metrics_rows,
    OUTPUT_DIR,
)

print("Saved metrics:")

for path in metric_files:
    print(path)

Saved metrics:
../../results_python_pipeline/outputs/metrics_summary.csv
../../results_python_pipeline/outputs/metrics_summary.json
